# 31 - The Agent Protocol Stack (Capstone)

## Scenario: The Unified Northstar Support OS

In the previous modules, we built individual components of an enterprise agent system (Routing, Guardrails, Memory, RBAC). 

In this Capstone, we combine **all of them** into a single, unified architecture: The Agent Protocol Stack.

1. **Routing Layer**: Decides if we need the expensive agent.
2. **Memory Layer**: Injects past context.
3. **Execution & RBAC Layer**: Runs the agent and blocks unauthorized tool calls.
4. **Guardrail Layer**: Scrubs the final output for PII.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Setting up the Layers

First, we define our individual enterprise layers.

In [2]:
import re
from pydantic import BaseModel

# --- 1. Memory Layer (Mocked Vector DB) ---
def query_memory(ticket: str) -> str:
    print("  🧠 [Memory Layer] Querying past incidents...")
    if "payment" in ticket.lower():
        return "Past Incident Context: Payment gateway was down yesterday. Issue refunds if requested."
    return "No relevant past incidents found."

# --- 2. RBAC Tool Layer ---
USER_ROLES = {"bob": "user", "admin_alice": "admin"}

def issue_refund_tool(user_id: str, amount: int):
    role = USER_ROLES.get(user_id, "guest")
    print(f"  🔒 [RBAC Layer] Verifying {user_id} (Role: {role}) for refund of ${amount}...")
    if role != "admin":
        return "ERROR: Only admins can issue refunds."
    return f"Refund of ${amount} processed successfully."

# --- 3. Guardrail Layer ---
def guardrail_scrub(text: str) -> str:
    print("  🛡️ [Guardrail Layer] Scrubbing PII...")
    ssn_pattern = r"\b\d{3}-\d{2}-\d{4}\b"
    return re.sub(ssn_pattern, "[REDACTED SSN]", text)


## 2. The Unified Execution Pipeline

Now we wire them all together.

In [3]:
class RouteDecision(BaseModel):
    complexity: str # "LOW" or "HIGH"

def unified_agent_os(user_id: str, ticket: str):
    print(f"\n==========================================")
    print(f"📩 Incoming Ticket from {user_id}: '{ticket}'")
    
    # 1. Routing Layer
    print("🚦 [Routing Layer] Using gpt-4o-mini to classify complexity...")
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": ticket}],
            response_format=RouteDecision
        )
        complexity = completion.choices[0].message.parsed.complexity
    except Exception:
        complexity = "HIGH" if "refund" in ticket.lower() else "LOW"
        
    if complexity == "LOW":
        print("⚡ [Fast Path] Routing to FAQ Bot.")
        raw_output = "Please check our FAQ for basic issues."
    else:
        print("🚀 [Heavy Path] Escalating to full Agent workflow.")
        
        # 2. Memory Layer
        context = query_memory(ticket)
        
        # 3. Execution & RBAC Layer (Simulated Agent Tool Call)
        print(f"  🤖 [Agent] I need to issue a refund based on memory: '{context}'")
        tool_result = issue_refund_tool(user_id, amount=50)
        
        raw_output = f"I investigated the issue. {tool_result} Also, the SSN on file is 123-45-6789."
        
    # 4. Guardrail Layer
    final_output = guardrail_scrub(raw_output)
    
    print(f"✅ Final Safe Output: {final_output}")
    print(f"==========================================\n")

# Scenario A: Bob tries to do something complex but lacks permissions
unified_agent_os("bob", "My payment failed, give me a refund!")

# Scenario B: Alice does the same thing, but has permissions
unified_agent_os("admin_alice", "My payment failed, give me a refund!")



📩 Incoming Ticket from bob: 'My payment failed, give me a refund!'
🚦 [Routing Layer] Using gpt-4o-mini to classify complexity...
🚀 [Heavy Path] Escalating to full Agent workflow.
  🧠 [Memory Layer] Querying past incidents...
  🤖 [Agent] I need to issue a refund based on memory: 'Past Incident Context: Payment gateway was down yesterday. Issue refunds if requested.'
  🔒 [RBAC Layer] Verifying bob (Role: user) for refund of $50...
  🛡️ [Guardrail Layer] Scrubbing PII...
✅ Final Safe Output: I investigated the issue. ERROR: Only admins can issue refunds. Also, the SSN on file is [REDACTED SSN].


📩 Incoming Ticket from admin_alice: 'My payment failed, give me a refund!'
🚦 [Routing Layer] Using gpt-4o-mini to classify complexity...
🚀 [Heavy Path] Escalating to full Agent workflow.
  🧠 [Memory Layer] Querying past incidents...
  🤖 [Agent] I need to issue a refund based on memory: 'Past Incident Context: Payment gateway was down yesterday. Issue refunds if requested.'
  🔒 [RBAC Layer] Verif

## Checkpoint

**1. What is the benefit of a layered Agent Protocol Stack over a monolithic prompt?**
- A) It is easier to write in one file.
- B) Separation of concerns. You can swap out the Memory DB, upgrade the Guardrail regex, or change the Routing model independently without breaking the entire agent.
- C) It is required by Python syntax.
- D) It reduces the number of files in the project.
